### Purpose

This notebook performs focused descriptive analytics on the approved Gold v1 data products.


The objective is to identify meaningful patterns and relationships that can be explored and presented through Power BI. This notebook does not modify Bronze, Silver, or Gold data and does not create additional Gold tables.

The analysis is descriptive. Observed relationships are not interpreted as causal or predictive unless explicitly supported by additional analysis.

### Analytical areas

The analysis focuses on four areas:

1. User behavior
- How concentrated are transaction activity and spending across users?
- What behavioral patterns are visible?
- How does observed fraud vary across users?

2. Card behavior
- How does activity vary across cards?
- How many cards are inactive?
- How does observed fraud vary at card level?

3. Merchant behavior
- How concentrated are transactions and spending across merchants?
- How does online behavior vary?
- How is observed fraud distributed across merchants?
- What geographic patterns are visible at transaction level?

4. Fraud features
- How does observed fraud relate descriptively to historical user fraud?
- How does observed fraud relate descriptively to historical card fraud?
- How does observed fraud relate descriptively to historical merchant fraud?
- Do historical transaction volumes differ between fraudulent and non-fraudulent transactions?

### Power BI objective

The outputs of this notebook will be used to identify a small number of meaningful measures, relationships, and visualizations for the Power BI analytics layer.

No unnecessary Gold tables or additional persistent data products will be created.

### Gold v1 data sources

The analysis uses the four approved Gold v1 data products:

| Gold object | Grain | Purpose |
|---|---|---|
| gold_user_behavior | user_id | User-level behavior |
| gold_card_behavior | (user_id, card_id) | Card-level behavior |
| gold_merchant_behavior | merchant_id | Merchant-level behavior |
| gold_fraud_features | One row per Silver transaction | Transaction-level fraud context |

The Gold tables are treated as the authoritative analytical inputs for this notebook.

## 2. Load Gold v1

In [0]:
# Load approved Gold v1 data products
user_df = spark.table("credit_card_project.gold.gold_user_behavior")
card_df = spark.table("credit_card_project.gold.gold_card_behavior")
merchant_df = spark.table("credit_card_project.gold.gold_merchant_behavior")
fraud_df = spark.table("credit_card_project.gold.gold_fraud_features")


In [0]:
#lightweight structural check
print("User behavior columns:")
print(user_df.columns)

print("\nCard behavior columns:")
print(card_df.columns)

print("\nMerchant behavior columns:")
print(merchant_df.columns)

print("\nFraud features columns:")
print(fraud_df.columns)


### 3. User Behavior

This section examines transaction activity, spending, and observed fraud at user level.

The analysis focuses on:
- distribution of transaction activity across users
- distribution of spending across users
- concentration of transactions and spending
- relationship between activity, spending, and observed fraud

The objective is to identify descriptive patterns that are useful for the Power BI analytics layer.

In [0]:
from pyspark.sql import functions as F

# user-level summary statistics
user_summary = user_df.select(
    F.count("*").alias("users"),
    F.sum("total_transactions").alias("transactions"),
    F.sum("total_spend").alias("total_spend"),
    F.avg("total_transactions").alias("avg_transactions_per_user"),
    F.expr("percentile_approx(total_transactions, 0.5)").alias("median_transactions_per_user"),
    F.avg("total_spend").alias("avg_spend_per_user"),
    F.expr("percentile_approx(total_spend, 0.5)").alias("median_spend_per_user"),
    F.avg("fraud_rate").alias("avg_user_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.5)").alias("median_user_fraud_rate")
)

display(user_summary)


In [0]:
#distribution of transaction activity
transaction_distribution = user_df.select(
    F.min("total_transactions").alias("min_transactions"),
    F.expr("percentile_approx(total_transactions, 0.25)").alias("p25_transactions"),
    F.expr("percentile_approx(total_transactions, 0.50)").alias("p50_transactions"),
    F.expr("percentile_approx(total_transactions, 0.75)").alias("p75_transactions"),
    F.expr("percentile_approx(total_transactions, 0.90)").alias("p90_transactions"),
    F.expr("percentile_approx(total_transactions, 0.95)").alias("p95_transactions"),
    F.expr("percentile_approx(total_transactions, 0.99)").alias("p99_transactions"),
    F.max("total_transactions").alias("max_transactions")
)

display(transaction_distribution)


In [0]:
#distribution of spending
spend_distribution = user_df.select(
    F.min("total_spend").alias("min_spend"),
    F.expr("percentile_approx(total_spend, 0.25)").alias("p25_spend"),
    F.expr("percentile_approx(total_spend, 0.50)").alias("p50_spend"),
    F.expr("percentile_approx(total_spend, 0.75)").alias("p75_spend"),
    F.expr("percentile_approx(total_spend, 0.90)").alias("p90_spend"),
    F.expr("percentile_approx(total_spend, 0.95)").alias("p95_spend"),
    F.expr("percentile_approx(total_spend, 0.99)").alias("p99_spend"),
    F.max("total_spend").alias("max_spend")
)

display(spend_distribution)


In [0]:
#calculate the cumulative share of transactions and spending contributed by the highest-activity users
from pyspark.sql import functions as F

# The user-level table contains only 2,000 rows,
# so collecting the small aggregated result is appropriate here.
user_concentration = (
    user_df
    .select(
        "user_id",
        "total_transactions",
        "total_spend"
    )
    .orderBy(F.desc("total_transactions"))
    .collect()
)

total_transactions = sum(row["total_transactions"] for row in user_concentration)
total_spend = sum(row["total_spend"] for row in user_concentration)

user_count = len(user_concentration)
concentration_levels = [1, 5, 10, 20]
rows = []

for pct in concentration_levels:
    n = max(1, int(user_count * pct / 100))
    top_users = user_concentration[:n]
    transaction_share = (
        sum(row["total_transactions"] for row in top_users)
        / total_transactions
    )

    top_spend_users = sorted(
        user_concentration,
        key=lambda row: row["total_spend"],
        reverse=True
    )[:n]

    spend_share = (
        sum(row["total_spend"] for row in top_spend_users)
        / total_spend
    )

    rows.append(
        (
            pct,
            n,
            transaction_share,
            spend_share
        )
    )

concentration_df = spark.createDataFrame(
    rows,
    [
        "top_user_pct",
        "user_count",
        "transaction_share",
        "spend_share"
    ]
)
display(concentration_df)


In [0]:
#user behavioral relationships: transactions vs. spending vs. fraud rate

user_relationships = user_df.select(
    "user_id",
    "total_transactions",
    "total_spend",
    "fraud_count",
    "fraud_rate",
    "unique_cards",
    "unique_merchants",
    "online_transaction_pct",
    "weekend_transaction_pct",
    "night_transaction_pct"
)

display(user_relationships)


In [0]:
#user behavioral relationships
import pandas as pd

correlation_columns = [
    "total_transactions",
    "total_spend",
    "fraud_count",
    "fraud_rate",
    "unique_cards",
    "unique_merchants",
    "online_transaction_pct",
    "weekend_transaction_pct",
    "night_transaction_pct"
]

user_corr_pd = (
    user_relationships
    .select(correlation_columns)
    .toPandas()
    .corr()
)

display(user_corr_pd)

What stands out:

A few relationships are strong enough to investigate further:
- Transactions ↔ spending: 0.851: Users with more transactions generally have higher total spending. This is unsurprising but useful for a Power BI scatter plot.

- Transactions ↔ unique merchants: 0.763: Higher-activity users tend to transact with more distinct merchants.

- Spending ↔ unique merchants: 0.787: This is another fairly strong relationship and potentially more interesting from a behavioral perspective.

- Fraud count ↔ fraud rate: 0.752: This needs some caution. fraud_rate is calculated from fraud count and transaction count, so a relationship between those variables is partly mathematical rather than an independent behavioral discovery.

- Transaction activity ↔ fraud rate: 0.102

- Spending ↔ fraud rate: 0.102: These are relatively weak relationships at user level. That's useful because it means we shouldn't build a story suggesting that high-volume/high-spend users necessarily have higher observed fraud rates.

- The weekend/night percentages have essentially no linear relationship with fraud rate in this user-level analysis.


What I'd carry forward to Power BI

I'd tentatively keep:
- User transaction volume vs. total spending — scatter plot
- User transaction volume vs. number of unique merchants — potentially useful
- User spending concentration — definitely useful
- Fraud rate distribution across users — useful, but probably as a distribution rather than a ranking

In [0]:
 #fraud-rate distribution to determine whether fraud is distributed among users or concentrated in a smaller population
fraud_rate_distribution = user_df.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.25)").alias("p25_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.50)").alias("p50_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.75)").alias("p75_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.90)").alias("p90_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.95)").alias("p95_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.99)").alias("p99_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate")
)

display(fraud_rate_distribution)


In [0]:
fraud_users = user_df.select(
    F.sum(
        F.when(F.col("fraud_count") > 0, 1).otherwise(0)
    ).alias("users_with_fraud"),
    F.count("*").alias("total_users")
)

display(fraud_users)
